# 01 · Demand Intelligence

**Forecast vs Instinct** - an FMCG demand-planning portfolio project built on **real data from Corporación Favorita,
an Ecuadorian grocery chain** (Kaggle *Store Sales - Time Series Forecasting*). Scope: **12 stores × 8 product
families = 96 daily series**.

This notebook tells the demand story before any model is built: how big each family is, which weekdays matter,
which series are hard to plan, and how sales move with promotions, holidays, the 2016 earthquake and paydays.

Three ground rules for reading everything below:

1. **Sales are observed sales, not unconstrained demand.** There is no stock-out data, so a day where shelves ran
   empty looks like a day with low demand.
2. **`onpromotion` is a count of promoted items** in a family/store/day. There is no price or discount depth.
   Every promotion number here is an **association, not a causal uplift**.
3. **No peeking at the test.** Every number uses history from **2015-06-01 up to 2017-06-27** (the last calibration
   target date). The holdout period (2017-06-28 → 2017-08-15) is never touched, so nothing here can leak into
   model evaluation.

All numbers come from the DuckDB SQL files in `sql/` (run through `src/eda.py`), the same queries that build
`outputs/eda/demand.json` for the dashboard, so the notebook and the site agree.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "sql").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from src import eda, config as C

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.precision", 3)

con = eda.connect()
q = lambda name, **kw: eda.run_sql(con, name, **kw)

# One fixed colour per family (never re-assigned when a chart shows fewer families)
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
FAM_COLOR = dict(zip(C.FAMILIES, PALETTE))
LAYOUT = dict(template="simple_white", height=380, margin=dict(l=50, r=20, t=50, b=40),
              font=dict(size=12), legend=dict(title=None))
print("EDA window:", eda.default_params()["start"], "->", eda.EDA_CUTOFF)

EDA window: 2015-06-01 -> 2017-06-27


## 1. What is in the data?

One row per store × family × day. A **closed** store-day (the chain closes on 25 December and 1 January) has
sales = *missing*, not zero: a closure is not zero demand, so it must not drag averages down or teach a model
that Christmas Day sells nothing.

In [2]:
ov = q("13_data_overview")
# Family volumes and volume shares are not shown: per-family unit levels are not published (Kaggle rules).
ov[["family", "n_stores", "n_store_days", "n_closed_store_days", "n_open_zero_days"]]

,family,n_stores,n_store_days,n_closed_store_days,n_open_zero_days
0,GROCERY I,12,9096,48.0,0
1,BEVERAGES,12,9096,48.0,0
2,PRODUCE,12,9096,48.0,0
3,CLEANING,12,9096,48.0,1
4,DAIRY,12,9096,48.0,0
5,BREAD/BAKERY,12,9096,48.0,0
6,PERSONAL CARE,12,9096,48.0,1
7,FROZEN FOODS,12,9096,48.0,1


**Reading it:** each family has 48 closed store-days (4 closure dates × 12 stores) and essentially no
zero-sales days on open days: the scope was chosen so that every series is recorded continuously.

**Index scale used from here on (Kaggle rule 7B).** Raw competition data is not redistributed, so every sales figure
below is an **index**: a series' daily sales ÷ that series' mean daily sales on open days in the training period
(2015-06-01 to 2017-02-06) × 100, so **100 = an average training-period day** for that store-family. Family and
chain lines are **equal-weighted** means of their member series' indexes (each store counts the same, whatever its
size), so no unit volumes can be recovered from them.

## 2. Why the window starts on 2015-06-01: the PRODUCE recording gap

Before mid-2015, PRODUCE shows **zero sales on roughly half to two-thirds of open store-days in many months**, while
the store was clearly trading (it sold other families). That is a **recording gap** (the family was not being
captured in many stores), not a lack of demand. From June 2015 the zero share drops to 0. Starting every series on
2015-06-01 gives one shared calendar and still leaves ~20 months of history before the first forecast origin.

*This check deliberately reads the raw pre-window history (2013-2015); it is a data-quality check, not a model input.*

In [3]:
scope = json.loads((C.CONFIG / "scope.json").read_text())
stores = [s["store_nbr"] for s in scope["stores"]]
gap = q("11_produce_recording_gap", stores=stores)
gap["group"] = np.where(gap["family"] == "PRODUCE", "PRODUCE", "Other 7 families (max)")
g = gap.groupby(["group", "month"], as_index=False)["zero_share"].max()
fig = px.line(g, x="month", y="zero_share", color="group", markers=True,
              color_discrete_map={"PRODUCE": FAM_COLOR["PRODUCE"], "Other 7 families (max)": "#8a8984"},
              title="Share of open store-days with zero recorded sales (12 scope stores)")
fig.add_vline(x=pd.Timestamp(C.MODEL_START).timestamp() * 1000, line_dash="dash", line_color="#52514e")
fig.add_annotation(x=pd.Timestamp(C.MODEL_START), y=1.02, text="window start 2015-06-01", showarrow=False, yanchor="bottom")
fig.update_layout(**LAYOUT, yaxis_tickformat=".0%", yaxis_title=None, xaxis_title=None)
fig.show()
pre = gap[pd.to_datetime(gap.month) < C.MODEL_START]
post = gap[pd.to_datetime(gap.month) >= C.MODEL_START]
w = lambda d: (d.zero_share * d.n_open_store_days).sum() / d.n_open_store_days.sum()
print(f"PRODUCE zero share on open store-days: before window {w(pre[pre.family=='PRODUCE']):.1%}, "
      f"in window {w(post[post.family=='PRODUCE']):.1%}")

PRODUCE zero share on open store-days: before window 48.3%, in window 0.0%


## 3. Size, trend and seasonality

Weekly **equal-weighted family index** (complete Monday-Sunday weeks only): for each store-family series, the mean
daily index over the week's open days; the family line is the plain mean of its 12 store series. 100 = an average
training-period day, so families of very different size can be compared on one axis.

In [4]:
d = eda.index_frame(con)          # open days, idx = sales / series base x 100 (base never shown)
weeks = sorted(q("01_weekly_family_totals")["week_start"].unique())
fwi = pd.DataFrame(eda.family_weekly_index(d, weeks), index=pd.to_datetime([str(w)[:10] for w in weeks]))
wk = fwi.rename_axis("week_start").reset_index().melt("week_start", var_name="family", value_name="index")
fig = px.line(wk, x="week_start", y="index", color="family", color_discrete_map=FAM_COLOR,
              category_orders={"family": C.FAMILIES},
              title="Weekly equal-weighted family index (100 = average training-period day)")
fig.update_traces(line_width=1.6)
fig.update_layout(**{**LAYOUT, "height": 430}, yaxis_title="index", xaxis_title=None)
fig.show()
allw = fwi.mean(axis=1).sort_values(ascending=False)   # equal-weighted over all 96 series
print("Top 5 weeks (equal-weighted index of all 96 series, relative to the median week):")
print((allw.head(5) / allw.median()).round(3).rename("x median week").to_string())

Top 5 weeks (equal-weighted index of all 96 series, relative to the median week):
2016-12-19    2.198
2015-12-21    1.889
2016-12-26    1.632
2015-12-14    1.505
2015-12-28    1.481


**Reading it:**

- **FROZEN FOODS is a December family.** Its index spikes in the weeks before Christmas far more than any other
  family. On average a FROZEN FOODS series sells 30 % of its in-window sales in December (next cell), against
  about 8-10 % for the other families.
- The single biggest week of the equal-weighted index is **the week of 19 December 2016** (2.2× the median week);
  the top five weeks are all Christmas / New Year weeks. The earthquake week (section 8) is a big week mainly for
  the large stores and the Manabí stores, so it ranks lower when every store counts the same.
- Most families grow between the first and the last 52 complete weeks (table below: DAIRY about +19 %, GROCERY I
  and BEVERAGES about +9-10 %); PERSONAL CARE is flat.

In [5]:
# December share per series (scale-free), then the plain mean over the family's 12 series
m = d.assign(dec=d.date.dt.month == 12)
ser = m.groupby(["family", "series_id"]).apply(lambda g: g.loc[g.dec, "sales"].sum() / g.sales.sum(), include_groups=False)
dec_share = ser.groupby("family").mean().rename("december_share").reset_index().sort_values("december_share", ascending=False)
ex = json.loads((C.OUTPUTS / "eda" / "eda_extras.json").read_text())
trend = pd.DataFrame(ex["trend_52w"])[["family", "first_52w_mean_index", "last_52w_mean_index", "change_pct"]]
dec_share.merge(trend, on="family").round(3)

,family,december_share,first_52w_mean_index,last_52w_mean_index,change_pct
0,FROZEN FOODS,0.300,95.92,101.80,6.12
1,GROCERY I,0.098,96.60,106.18,9.91
2,BEVERAGES,0.089,98.04,106.66,8.79
3,CLEANING,0.088,98.68,103.37,4.75
4,PRODUCE,0.088,98.41,104.27,5.96
5,PERSONAL CARE,0.086,100.04,99.58,-0.46
6,DAIRY,0.085,93.00,110.56,18.88
7,BREAD/BAKERY,0.077,98.47,99.99,1.54


*(The December share is computed per series on the same window and then averaged over each family's 12 series.)*

## 4. Which weekdays drive volume?

For each series: mean index on each weekday; the family value is the plain mean over its 12 series, normalised so
the 7 values average exactly 1.0. National days off are left out so holidays don't distort the pattern.

In [6]:
dow = eda.equal_weighted_dow(d)
dmat = pd.DataFrame(dow, index=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]).T.loc[C.FAMILIES]
fig = px.imshow(dmat.round(2), text_auto=True, aspect="auto", color_continuous_scale="RdBu_r",
                zmin=0.6, zmax=1.4, title="Day-of-week index (1.00 = average day, equal-weighted over stores)")
fig.update_layout(**{**LAYOUT, "height": 400}, coloraxis_colorbar=dict(title=None))
fig.show()

**Reading it:** the weekend (especially **Sunday**) is the peak for almost every family and **Thursday is the
quietest day**. FROZEN FOODS peaks on Saturday. **PRODUCE is the outlier: its busiest days are mid-week, Tuesday above all
(Wednesday is also above average).**
The next table shows that PRODUCE has a strong weekday *promotion* rhythm: on Tuesdays and Wednesdays the promo index
(promoted-item count relative to each series' own mean) is many times higher than on other days. The mid-week peak is therefore **associated with**
a mid-week produce promotion pattern; the data cannot tell us how much of the peak the promotions cause.

In [7]:
# promo index = promoted-item count / the series' mean count up to the cutoff x 100 (raw counts not shown)
wp = pd.DataFrame(ex["weekday_promo_profile"])
wp[wp.family == "PRODUCE"][["dow", "mean_promo_index", "mean_index", "n_store_days"]]

,dow,mean_promo_index,mean_index,n_store_days
49,1,12.94,90.27,1308
50,2,364.99,124.89,1308
51,3,263.96,105.77,1296
52,4,12.50,79.31,1296
53,5,13.15,84.64,1272
54,6,13.71,108.88,1296
55,7,13.90,115.18,1272


## 5. Which series are hard to plan?

- **CV (coefficient of variation)** = standard deviation ÷ mean. Daily CV includes the weekday swing, so we also
  show **weekly CV** (on complete-week totals), which is closer to how a planner feels volatility.
- **Zero share** = share of open days with zero sales (≈ 0 everywhere in this scope).

In [8]:
vol = q("03_volatility_by_series").drop(columns="mean")  # per-series levels are not shown (Kaggle rules)
fig = px.strip(vol, x="family", y="cv_weekly", hover_data=["store_nbr", "cv"],
               category_orders={"family": C.FAMILIES}, color_discrete_sequence=["#2a78d6"],
               title="Weekly CV per store-family series (each dot = one of 96 series)")
fig.update_traces(marker_size=9)
fig.update_layout(**LAYOUT, xaxis_title=None, yaxis_title="weekly CV")
fig.show()
vol.sort_values("cv_weekly", ascending=False).head(8)[["store_nbr", "family", "cv", "cv_weekly", "zero_share"]]

,store_nbr,family,cv,cv_weekly,zero_share
52,38,FROZEN FOODS,3.168,1.803,0.001
12,14,FROZEN FOODS,2.218,0.783,0.000
86,53,PERSONAL CARE,1.308,0.730,0.001
68,49,FROZEN FOODS,1.108,0.651,0.000
36,31,FROZEN FOODS,2.042,0.648,0.000
84,53,FROZEN FOODS,1.363,0.625,0.000
76,51,FROZEN FOODS,0.996,0.563,0.000
28,29,FROZEN FOODS,1.247,0.537,0.000


**Reading it:** **FROZEN FOODS is in a volatility class of its own** - 7 of the 8 most volatile series (weekly CV)
are FROZEN FOODS, driven by short, very large December peaks (one store's single biggest day is more than 50× its
average day). The exception in the top 8 is PERSONAL CARE in store 53 (Manta, Manabí), a store whose sales level
shifted after the 2016 earthquake (section 8). The other families sit in a tight band of weekly CV around 0.1-0.2. Expect forecast error to
concentrate in FROZEN FOODS around December, and expect intervals there to be wide.

## 6. Promotions (association, not causation)

**How often are series on promotion?** Promotions are close to permanent for GROCERY I and BEVERAGES, so "promo vs
no promo" is not a meaningful contrast for them; what varies is *how many* items are promoted.

In [9]:
pf = q("05_promo_frequency")
pf.groupby("family")[["promo_day_share", "corr_promo_sales"]].median().loc[C.FAMILIES].round(3)

,promo_day_share,corr_promo_sales
family,,
BEVERAGES,0.969,0.183
DAIRY,0.804,0.139
BREAD/BAKERY,0.759,0.379
GROCERY I,0.989,0.202
FROZEN FOODS,0.654,0.169
CLEANING,0.937,0.076
PERSONAL CARE,0.770,0.087
PRODUCE,0.637,0.619


**How do sales vary with the number of promoted items?** The raw mean rises with the promo count for most
families, but this mixes two things: big stores promote more items *and* sell more. The **mean index** compares
each day with the same store-family's average for that calendar month, which removes store size and slow trend.
`n_days` shows how much data sits behind each bar; bars with few stores are fragile.

In [10]:
pb = q("04_promo_bins")
fig = px.bar(pb, x="bin", y="mean_index", facet_col="family", facet_col_wrap=4, color="family",
             color_discrete_map=FAM_COLOR, category_orders={"family": C.FAMILIES, "bin": ["0", "1-5", "6-15", "16-30", "31-60", "61+"]},
             hover_data=["n_days", "n_stores"],
             title="Sales index by number of promoted items (1.0 = store-family's monthly average)")
fig.add_hline(y=1.0, line_dash="dot", line_color="#52514e")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(**{**LAYOUT, "height": 520}, showlegend=False)
fig.update_xaxes(title=None)
fig.update_yaxes(title=None)
fig.show()
pb.pivot(index="family", columns="bin", values="mean_index")[["0", "1-5", "6-15", "16-30", "31-60", "61+"]].round(3)

bin,0,1-5,6-15,16-30,31-60,61+
family,,,,,,
BEVERAGES,0.979,0.992,0.966,1.019,1.023,1.034
BREAD/BAKERY,0.964,0.968,0.974,0.990,1.308,1.348
CLEANING,0.976,0.999,1.005,1.002,1.006,1.038
DAIRY,1.014,0.994,0.993,1.007,1.004,0.989
FROZEN FOODS,0.896,0.953,1.378,1.681,1.527,NaN
GROCERY I,0.998,0.970,1.007,0.989,0.998,1.018
PERSONAL CARE,0.988,0.984,1.010,1.082,1.035,1.040
PRODUCE,0.916,0.914,0.935,0.890,1.102,1.494


**Reading it:** once store size and month are controlled for, the association between the promoted-item count and
sales is **small for GROCERY I, BEVERAGES, CLEANING and DAIRY** (within a few percent of the monthly average across
bins). It is **much larger for PRODUCE (61+ items: index ~1.5)** and **FROZEN FOODS (6-30 items: index ~1.4-1.7)**; BREAD/BAKERY's top bins (31+ items) are also ~30 % higher, but only 5-7 stores ever reach them.
These are associations: PRODUCE's big promotions fall on Tuesdays/Wednesdays (section 4), and FROZEN FOODS
promotions cluster in December, so part of each pattern is calendar, not promotion. Without prices or discount
depth this data cannot measure promotional uplift.

## 7. Holidays

Holiday flags are matched to each store: **National** holidays apply to all stores, **Regional** ones to stores in
that state, **Local** ones to stores in that city. Rules from the Kaggle file:

- a `Holiday` that was **transferred** is *not* a day off; the day it moved to (`Transfer`) is;
- `Bridge` and `Additional` days are extra days off; a `Work Day` is a weekend day made a working day;
- `Event` rows (Black Friday, Mother's Day, the earthquake aftermath, ...) are not days off.

**Simple comparison** (national days off vs normal days, `06_holiday_effect.sql`). Normal days exclude any day off
of any locale for that store, Work Days, Events and the earthquake month.

In [11]:
he = q("06_holiday_effect")
# % changes and counts only: holiday / normal-day sales levels are not published
he[["family", "change_pct", "change_pct_excl_additional", "n_holiday_days", "n_holiday_days_excl_additional"]].round(2)

,family,change_pct,change_pct_excl_additional,n_holiday_days,n_holiday_days_excl_additional
0,BEVERAGES,19.41,10.96,34,20
1,BREAD/BAKERY,6.28,5.58,34,20
2,CLEANING,20.28,12.26,34,20
3,DAIRY,20.18,11.11,34,20
4,FROZEN FOODS,226.90,-0.62,34,20
5,GROCERY I,27.56,8.84,34,20
6,PERSONAL CARE,19.36,17.28,34,20
7,PRODUCE,13.97,1.58,34,20


**Reading it:** the raw comparison is dominated by the national **Additional** days, which are mostly the Christmas
run-up (21-26 and 31 December). That is why FROZEN FOODS looks like +200 %. Removing them, national days off are
associated with about −1 % to +17 % higher sales per store-day depending on family. These raw means still ignore
the weekday mix (a holiday on a Monday vs a Saturday), which the next analysis fixes.

### Ecuador holiday analogue (weekday-matched)

Used by the scenario lab as a **real-data analogue** for event assumptions. Precise definition
(`10_ecuador_holiday_analogue.sql`):

- **Holiday dates:** national `Holiday` (not transferred), `Transfer` and `Bridge` days in the window. Additional
  days, Events and Work Days are not holiday dates.
- **Baseline:** for each store, the mean of sales on the **same weekday 1 and 2 weeks before and after**
  (t−14, t−7, t+7, t+14), keeping only baseline days that are open, inside the window, and not a day off / Work Day
  / Event of any locale for that store, and outside the earthquake month. A store needs ≥ 2 valid baseline days.
- **Per holiday × family:** % change = Σ stores sales on the day ÷ Σ the same stores' baseline − 1.
- **Family figure:** the simple mean over holidays. Holidays when the chain is closed (25 Dec, 1 Jan) drop out,
  and 1 May 2016 drops out because all its baseline days fall in the earthquake month.

In [12]:
import re
d0 = q("10_ecuador_holiday_analogue", offset=0)
summ = pd.DataFrame(eda.analogue_summary(d0)).drop(columns="window")
display(summ.round(2))
# Per-holiday detail, EQUAL-WEIGHTED: % change per store-family series, then the plain mean over the family's stores
sql = eda.sql_text("10_ecuador_holiday_analogue")
sql = sql[:sql.rindex("SELECT holiday_date")] + (
    "SELECT holiday_date, description, store_nbr, family, 100 * (target_sales / baseline_mean - 1) AS change_pct FROM j")
prm = {**eda.default_params(), "offset": 0}
per = con.execute(sql, {k: prm[k] for k in set(re.findall(r"\$([A-Za-z_]\w*)", sql))}).df()
per["label"] = pd.to_datetime(per.holiday_date).dt.strftime("%Y-%m-%d") + " " + per.description.str.slice(0, 32)
mat = per.groupby(["label", "family"]).change_pct.mean().unstack()[C.FAMILIES]
fig = px.imshow(mat.round(0), text_auto=True, aspect="auto", color_continuous_scale="RdBu_r", zmin=-60, zmax=60,
                title="% change vs same-weekday baseline, per national holiday (mean over the family's stores)")
fig.update_layout(**{**LAYOUT, "height": 640}, coloraxis_colorbar=dict(title="%"), yaxis_title=None, xaxis_title=None)
fig.show()

,family,change_pct,n_holidays,median_pct,min_pct,max_pct
0,BEVERAGES,17.25,19,17.26,-24.94,65.16
1,DAIRY,14.67,19,15.23,-22.85,57.05
2,BREAD/BAKERY,9.49,19,8.24,-23.88,51.36
3,GROCERY I,11.91,19,12.29,-28.95,54.78
4,FROZEN FOODS,26.54,19,31.04,-47.16,56.78
5,CLEANING,16.96,19,10.27,-22.37,66.70
6,PERSONAL CARE,27.82,19,17.04,-35.01,88.71
7,PRODUCE,7.63,19,6.52,-20.53,44.22


**Reading it:** on average, national days off are **associated with higher sales than the same weekday in the
surrounding weeks for every family**, but the **average hides a wide spread**:

- **Carnaval Monday** and **Viernes Santo (Good Friday)** are associated with *lower* sales for most families;
- **Dia del Trabajo (1 May 2017)** and the **day after New Year (2 Jan 2017)** sit far above their baselines;
- FROZEN FOODS on 2 Jan 2017 is strongly negative, because its baseline (mid-December) is its seasonal peak.

So the family-level average is a sensible *starting assumption* for an unknown event, but a planner should expect
individual holidays to land well above or below it (see the min/max columns).

In [13]:
dm1 = q("10_ecuador_holiday_analogue", offset=-1)
pd.DataFrame(eda.analogue_summary(dm1, window="day before")).drop(columns="window").round(2)

,family,change_pct,n_holidays,median_pct,min_pct,max_pct
0,BEVERAGES,-6.78,13,-2.57,-33.14,18.56
1,DAIRY,-10.39,13,-6.71,-36.80,13.04
2,BREAD/BAKERY,-11.76,13,-10.71,-33.27,4.70
3,GROCERY I,-6.63,13,-6.18,-28.87,27.97
4,FROZEN FOODS,15.95,13,7.83,-6.54,51.69
5,CLEANING,-4.49,13,-3.19,-25.08,31.46
6,PERSONAL CARE,-10.87,13,-9.60,-31.28,44.35
7,PRODUCE,-8.55,13,-2.22,-32.18,13.63


The **day before** a national holiday (same method, target = h−1, skipping targets that are themselves days off)
is, on average, associated with *lower* sales than its weekday baseline for most families, with FROZEN FOODS the
exception. Treat this as descriptive: it is a small sample of 13 usable eves.

## 8. The 16 April 2016 earthquake

A magnitude-7.8 earthquake struck Manabí province on 16 April 2016. Two scope stores are in Manabí
(**53 Manta** and **54 El Carmen**). We compare the **equal-weighted mean index** (per series mean daily index, then
the plain mean over the group's series, all 8 families) in the 30 days before, the 30 days from the quake (to
15 May) and the 30 days after.

In [14]:
a = pd.DataFrame(ex["earthquake"]["all_families"]).set_index("store_group")
display(a[["pre", "quake", "post", "quake_vs_pre_pct", "post_vs_pre_pct"]].round(1))
dq = eda.index_frame(con, cutoff="2016-07-31")
dq = dq[dq.date >= "2016-02-01"].assign(grp=lambda x: np.where(x.state == "Manabi", "Manabi stores (53, 54)", "Other 10 stores"))
wq = dq.groupby(["grp", "week_start", "series_id"]).idx.mean().groupby(["grp", "week_start"]).mean().reset_index()
base = wq[wq.week_start < "2016-04-11"].groupby("grp").idx.mean()
wq["index"] = 100 * wq.idx / wq.grp.map(base)
fig = px.line(wq, x="week_start", y="index", color="grp", markers=True,
              color_discrete_sequence=["#e34948", "#2a78d6"],
              title="Weekly equal-weighted sales index, rebased to Feb-early Apr 2016 average (= 100)")
fig.add_vrect(x0="2016-04-16", x1="2016-05-15", fillcolor="#8a8984", opacity=0.15, line_width=0,
              annotation_text="quake month", annotation_position="top left")
fig.update_layout(**LAYOUT, xaxis_title=None, yaxis_title="index")
fig.show()

,pre,quake,post,quake_vs_pre_pct,post_vs_pre_pct
store_group,,,,,
Manabi,80.8,128.3,140.5,58.8,73.9
Other,90.8,107.2,94.4,18.0,4.0


**Reading it:**

- **Other 10 stores:** a sharp, temporary jump. The quake month averages about 18 % above the month before, then
  sales fall back to within about 4 % of the pre-quake level. This is consistent with relief buying and donations nationwide.
- **Manabí stores:** a step change. Sales in the quake month were about 59 % above the month before and stayed
  high afterwards (the 30 days after are about 74 % above the pre-quake month). The data cannot tell us why
  (possible reasons include reconstruction activity or damaged competing stores); it is one event with no control group.

*The weekly chart and the table use the equal-weighted index (`src/eda.py`, outputs/eda/eda_extras.json).*
For modelling, the quake month is flagged (`quake`) and excluded from every baseline in this notebook.

## 9. Paydays

Ecuadorian public-sector wages are paid on the **15th and the last day of the month**. To avoid confusing a payday
with a weekday or a trend, each day is compared with the **same store-family, same month, same weekday** on
non-payday days (index 1.0 = normal). Days off, Work Days, Events and the earthquake month are excluded.

In [15]:
pay = q("08_payday_effect")
dom = pay[pay.family.isna() & pay.day_of_month.notna()].sort_values("day_of_month")
fig = px.bar(dom, x="day_of_month", y="mean_index", color_discrete_sequence=["#2a78d6"],
             hover_data=["n_store_days"], title="Sales index by day of month (all 8 families, 12 stores)")
fig.add_hline(y=1.0, line_dash="dot", line_color="#52514e")
fig.update_layout(**LAYOUT, xaxis_title="day of month", yaxis_title="index", yaxis_range=[0.85, 1.2])
fig.show()
pay[pay.family.notna()].pivot(index="family", columns="day_class", values="mean_index").round(3)

day_class,day_after,other,payday
family,,,
BEVERAGES,1.069,0.995,1.011
BREAD/BAKERY,1.069,0.995,1.009
CLEANING,1.097,0.993,0.994
DAIRY,1.062,0.995,1.004
FROZEN FOODS,1.028,0.998,1.076
GROCERY I,1.094,0.993,1.011
PERSONAL CARE,1.138,0.990,0.992
PRODUCE,1.048,0.996,1.000


**Reading it:** the clearest pattern is **the start of the month**: the 1st-5th sit roughly 8-16 % above a normal
same-weekday day, tapering through the first week, and the second week (8th-14th) dips below normal. The 15th itself
is not a stand-out day for most families, but the 16th and the 1st ("day after payday") are associated with higher
sales (index ~1.03-1.14 depending on family). Month-end salaries and other income timing all overlap here, so this is
an association with the calendar, not proof of a payday mechanism.

## 10. Transactions vs sales (EDA only)

`transactions` = number of receipts per store-day. It tracks sales closely, but it is **not used as a model
feature**: on the day we make a forecast, future transactions are unknown, so using them would be leakage.

In [16]:
tx = q("09_transactions_vs_sales")
pooled = tx[tx.store_nbr.isna()].iloc[0]
print(f"Pooled daily correlation (12 stores, sales of the 8 families vs transactions): {pooled.corr_daily:.3f} "
      f"over {int(pooled.n_days):,} store-days; {int(pooled.n_missing_txn)} store-days have no transactions record.")
ts = tx[tx.store_nbr.notna()].copy()
ts["store"] = ts.store_nbr.astype(int).astype(str)
fig = px.bar(ts.sort_values("corr_daily"), x="corr_daily", y="store", orientation="h",
             color_discrete_sequence=["#2a78d6"], title="Daily correlation of transactions with sales, per store")
fig.update_layout(**{**LAYOUT, "height": 420}, xaxis_range=[0, 1], xaxis_title="Pearson r", yaxis_title="store")
fig.show()

Pooled daily correlation (12 stores, sales of the 8 families vs transactions): 0.897 over 9,025 store-days; 23 store-days have no transactions record.


**Reading it:** footfall and sales move together strongly in every store (pooled correlation ≈ 0.9; per store
≈ 0.73-0.92). Part of this is shared weekly and seasonal rhythm. Useful for understanding demand; not available at
forecast time.

## 11. Oil price (context)

Ecuador is an oil exporter, and the oil price collapsed in 2015-2016. Both lines are indexed to June 2015 = 100.
This is context only: oil is not a model feature, and two trending series will look related whether or not one
drives the other.

In [17]:
oil = q("12_oil_vs_sales_monthly").dropna(subset=["oil_mean"])
mi = d.groupby([d.date.dt.to_period("M").dt.to_timestamp(), "series_id"]).idx.mean().groupby(level=0).mean()
oil["sales_index"] = pd.to_datetime(oil.month).map(mi)
o = pd.DataFrame({"month": oil.month,
                  "WTI oil price": 100 * oil.oil_mean / oil.oil_mean.iloc[0],
                  "Sales (equal-weighted index, 96 series)": 100 * oil.sales_index / oil.sales_index.iloc[0]}
                 ).melt("month", var_name="series", value_name="index")
fig = px.line(o, x="month", y="index", color="series", markers=True, color_discrete_sequence=["#52514e", "#2a78d6"],
              title="Oil price and sales, indexed to June 2015 = 100")
fig.update_layout(**LAYOUT, xaxis_title=None, yaxis_title="index")
fig.show()
print(f"Oil: {oil.oil_mean.iloc[0]:.2f} USD (Jun 2015) -> low {oil.oil_mean.min():.2f} "
      f"({pd.Timestamp(oil.loc[oil.oil_mean.idxmin(), 'month']):%b %Y}) -> {oil.oil_mean.iloc[-1]:.2f} (Jun 2017)")

Oil: 59.82 USD (Jun 2015) -> low 30.32 (Feb 2016) -> 45.17 (Jun 2017)


## 12. What this means for the forecast

| Finding | Implication for planning / modelling |
|---|---|
| Sunday peak, Thursday trough; PRODUCE peaks mid-week (Tuesday) | Weekly seasonality is essential; seasonal-naive is a strong baseline. |
| FROZEN FOODS is December-driven and by far the most volatile | Expect the widest intervals and most exceptions there; December is out of the holdout, but the history carries it. |
| Promotion count: small association for big families once store size and month are controlled; larger for PRODUCE and FROZEN FOODS | Promo count is worth testing as a feature (variant A vs B), but there is no basis to claim causal uplift. |
| National days off: positive on average, very mixed by holiday | Holiday flags help; one-size uplift assumptions are risky. |
| Earthquake: temporary national spike, lasting level shift in Manabí | Flag the quake month; Manabí stores have a structural break. |
| Start-of-month lift | Day-of-month / payday features are reasonable candidates. |
| Transactions track sales | Useful for diagnosis only: not known in advance. |

All numbers above come from history up to 2017-06-27 and from observed (not unconstrained) sales.